# 06 · Why is the policy trained on our demos worse than the one trained on teleoperation?

Three diagnostics, none of which trains anything (checkpoints come from notebook 03 via the Hub):

* **A. Failure breakdown of the offline-RL pair.** The two `ours_mix` runs (advantage-weighted 66%, unweighted 54%) were evaluated before `act.py` recorded where the bowl ended. Re-evaluating them sorts their failures into *never lifted / left off the plate / on the plate but > 3 cm from its centre*, like the other runs in the README. This says *where* the weighting helps: grasping or placing.
* **B. Execution horizon (optional, hypothesis H5).** ACT predicts 50 actions and executes 25 of them without looking (1.25 s). Re-evaluate the same checkpoints executing only 10 (0.5 s), and with temporal ensembling (re-plan every step and average the overlapping predictions). If `ours_all` improves much more than `official`, open-loop execution during the precise rim grasp is part of the gap.
* **C. How consistent are the demonstrations? (hypothesis H2).** Wrist angle and timing at the moment of grasp, across the episodes of `official` and `ours_all`. CPU only, a minute.

**Runtime:** T4 GPU (rendering). Each 50-episode evaluation takes ~5-10 min: A ≈ 15 min, B ≈ 40 min, C ≈ 1 min. Note that GPU evaluation is not bit-reproducible: a re-evaluation of the same checkpoint differs by a few episodes (see the README), so compare rates with their intervals.

In [ ]:
import os, sys, json, subprocess
os.environ["MUJOCO_GL"] = "egl"                      # GPU rendering, must be set before mujoco is imported
REPO_URL = "https://github.com/LukasWill/phone2panda.git"
BASE = "/content" if os.path.exists("/content") else "/kaggle/working"   # Colab or Kaggle
CODE_DIR = f"{BASE}/phone2panda"
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[libero]==0.6.1", "imageio-ffmpeg"], check=True)
# Colab preinstalls an older torchao; a newer diffusers (pulled in by lerobot) then fails on import
# (FqnToConfig). ACT does not use torchao, and diffusers skips it when it is absent.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
from p2p.sim.env import ensure_nvidia_egl
ensure_nvidia_egl()      # Colab: make EGL render on the GPU (otherwise it silently renders on the CPU, ~10x slower)
# Hugging Face token (Colab: key icon -> Secrets -> HF_TOKEN; Kaggle: Add-ons -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: uploads to the Hub will fail")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    print("Hugging Face account:", whoami(token=os.environ["HF_TOKEN"])["name"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
HUB_REPO = "phone2panda-data"
RERUN = ["act_ours_mix_awr_s0", "act_ours_mix_s0"]      # A: advantage-weighted and unweighted ours_mix
ABLATE = ["act_ours_all_s0", "act_official_s0"]          # B: execution-horizon ablation (official = control)
RUN_ABLATION = True                                      # B takes ~40 min; set False to skip it
OUT = f"{BASE}/diag"
from p2p.hub import pull, push
for run in RERUN + (ABLATE if RUN_ABLATION else []):
    pull(HUB_REPO, f"{BASE}/hub", allow_patterns=[f"runs/{run}/final/*"])

## A · Re-evaluate the two `ours_mix` checkpoints (same 50 scenes, now with the bowl's fate)

In [ ]:
for run in RERUN:
    !python -m p2p.policy.act eval {BASE}/hub/runs/{run}/final --episodes 50 --videos 0 --out {OUT}/{run}/eval_rerun

## B · Execution horizon (optional)
`--n-action-steps 10`: execute 10 of the 50 predicted actions, then predict again. `--temporal-ensemble 0.01`: predict every step and execute a weighted average of all predictions made for this step (ACT's original inference mode; `0.01` is the paper's weighting).

In [ ]:
if RUN_ABLATION:
    for run in ABLATE:
        !python -m p2p.policy.act eval {BASE}/hub/runs/{run}/final --episodes 50 --videos 0 --n-action-steps 10 --out {OUT}/{run}/eval_n10
        !python -m p2p.policy.act eval {BASE}/hub/runs/{run}/final --episodes 50 --videos 0 --temporal-ensemble 0.01 --out {OUT}/{run}/eval_te

## C · Demonstration consistency: wrist angle and timing at the grasp

In [ ]:
pull(HUB_REPO, f"{BASE}/hub", allow_patterns=[f"lerobot/{d}/{p}/*" for d in ("official", "ours_all") for p in ("data", "meta")])
!python tools/demo_consistency.py {BASE}/hub/lerobot/official {BASE}/hub/lerobot/ours_all | tee {OUT}/demo_consistency.jsonl

## Summary table (paste into the README) and save

In [ ]:
import glob
files = sorted(glob.glob(f"{OUT}/*/eval*/eval.json"))
!python tools/policy_failures.py {" ".join(files)} | tee {OUT}/policy_failures.md

In [ ]:
push(OUT, HUB_REPO, path_in_repo="diag")
!cd {BASE} && zip -qr diag.zip diag -x "*.mp4"
try:
    from google.colab import files; files.download(f"{BASE}/diag.zip")
except Exception:
    print("download", f"{BASE}/diag.zip", "from the Output panel")